 #  2. Distribution gallery

 In this notebook, we introduce the **distribution gallery** in CUQIpy: a collection of two-dimensional "toy" distributions provided for illustrative purposes and for testing and benchmarking samplers. Each gallery distribution is specified by its log-density (and gradient) only, making them ideal test targets for the sampling methods covered later in the book. The set follows the benchmark distributions of [The Markov-chain Monte Carlo Interactive Gallery](https://github.com/chi-feng/mcmc-demo).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cuqi
from cuqi.distribution import DistributionGallery
from cuqi.sampler import MH, NUTS
np.random.seed(0)

 ### 2.1. The gallery at a glance

 The gallery is loaded through `DistributionGallery`, which takes the name of the desired distribution as a string. The following distributions are available:

 | Name | Shape | Useful for testing |
 |---|---|---|
 | `"CalSom91"` | Two crescent-shaped modes | Multimodality with curved modes |
 | `"BivariateGaussian"` | Correlated Gaussian ($\rho = 0.9$) | Baseline; the only gallery distribution with direct sampling |
 | `"funnel"` | Neal's funnel | Distributions whose scale varies strongly across the space |
 | `"mixture"` | Mixture of three Gaussians | Multimodality |
 | `"squiggle"` | Strongly correlated, "squiggly" Gaussian | Strong correlation |
 | `"donut"` | Ring of radius $r$ | Non-convex support and strong curvature |
 | `"banana"` | Twisted ("banana-shaped") Gaussian | Strong curvature; a classic benchmark for HMC-type samplers |

 All gallery distributions share the same interface: they provide a `logpdf` (and a `gradient`), and most of them are *density-only* — that is, they cannot be sampled directly, only via a sampler.

 ### 2.2. Loading a distribution: the "donut"

 As a first example, we load the "donut" distribution, which is a bivariate distribution of a donut shape. Its log-density is

 $$
 \begin{aligned}
 \log(p(\mathbf{x})) \propto - \frac{1}{\sigma_\text{donut}^2} \left( \left\| \mathbf{x} \right\| - r_\text{donut} \right)^2,
 \end{aligned}
 $$

 where $\mathbf{x} = (x_1, x_2)$ is a 2D vector, $\left\| \mathbf{x} \right\|$ is the Euclidean norm of $\mathbf{x}$, $r_\text{donut}$ is the radius of the donut, and $\sigma_\text{donut}$ is a scalar that controls the width of the "donut". The density is largest where $\left\| \mathbf{x} \right\| \approx r_\text{donut}$ — i.e. on a ring — and the default parameters are $r_\text{donut} = 2.6$ and $\sigma_\text{donut}^2 = 0.033$.

In [ ]:
target_donut = DistributionGallery("donut")

print(target_donut)

 ### 2.3. Plotting the densities

 CUQIpy provides `cuqi.utilities.plot_2D_density` for visualizing 2D densities. It works directly on any 2D CUQIpy density, including the gallery distributions:

In [ ]:
cuqi.utilities.plot_2D_density(target_donut, -4, 4, -4, 4)
plt.title("The donut distribution")
plt.show()

 Let's also plot a few more gallery distributions. Note the different
 plotting windows needed to capture the mass of each distribution.

In [ ]:
for name, window in [("banana", (-5, 5)), ("funnel", (-8, 8)), ("mixture", (-4, 4))]:
    target = DistributionGallery(name)
    cuqi.utilities.plot_2D_density(target, window[0], window[1], window[0], window[1])
    plt.title(f"The {name} distribution")
    plt.show()

 The plots show the variety of shapes in the gallery: the banana is a strongly curved Gaussian, the funnel has a narrow "neck" (the width of $x_1$ varies by orders of magnitude depending on $x_2$), and the mixture is clearly multimodal.

 ### 2.4. Density and gradient evaluation

 All gallery distributions provide `logpdf` and `gradient`, which is the interface CUQIpy samplers rely on:

In [ ]:
x = np.array([2.0, 1.0])
print("logpdf at", x, ":", target_donut.logpdf(x))
print("gradient at", x, ":", target_donut.gradient(x))

 Note that most gallery distributions do *not* implement direct sampling (an exception is `"BivariateGaussian"`, which wraps a `Gaussian`). This is intentional: the gallery targets are meant to be explored with the samplers, which we do next.

In [ ]:
target_bivariate = DistributionGallery("BivariateGaussian")
print("BivariateGaussian can be sampled directly:", target_bivariate.sample(3))

 ### 2.5. Sampling from the gallery

 Since the gallery targets are density-only, we sample them with CUQIpy samplers. The sampling workflow is: construct the sampler with an initial point, `warmup` (adapt), then `sample`, and finally collect the samples with `get_samples`.

 Below we sample the donut distribution with two different samplers: Metropolis–Hastings (`MH`) with a fixed proposal scale, and the No-U-Turn Sampler (`NUTS`), which adapts its step size and uses the gradient.

In [ ]:
# Metropolis-Hastings with a fixed proposal scale
sampler_mh = MH(target_donut, scale=0.3, initial_point=np.array([3.0, 0.0]))
sampler_mh.warmup(200)
sampler_mh.sample(1000)
samples_mh = sampler_mh.get_samples()

In [ ]:
# NUTS (gradient-based, self-adapting)
sampler_nuts = NUTS(target_donut, initial_point=np.array([3.0, 0.0]))
sampler_nuts.warmup(200)
sampler_nuts.sample(500)
samples_nuts = sampler_nuts.get_samples()

In [ ]:
# Overlay the samples on the density
cuqi.utilities.plot_2D_density(target_donut, -4, 4, -4, 4)
plt.scatter(samples_mh.samples[0, :200], samples_mh.samples[1, :200],
            s=4, alpha=0.5, color="tab:red", label="MH")
plt.scatter(samples_nuts.samples[0, :200], samples_nuts.samples[1, :200],
            s=4, alpha=0.5, color="tab:blue", label="NUTS")
plt.legend()
plt.title("Samples from the donut distribution")
plt.show()

 Both samplers correctly concentrate the samples on the ring. The `MH` sampler with a small fixed step size moves slowly around the ring, while `NUTS` (which adapts its step size and follows the gradient) explores it more efficiently. The donut is a nice illustration of why the choice of sampler and its tuning parameters matter. For more details on sampling the donut distribution, we refer to {ref}`sampling-with-cuqipy`.

 ### 2.6. Exercises

 :::{admonition} **Exercises**
 :class: tip

 1. Plot the densities of two gallery distributions that were not shown above (`"CalSom91"`, `"squiggle"`, or `"BivariateGaussian"`). Experiment with the plotting window to capture the mass of the distribution.
 2. The `"BivariateGaussian"` distribution is the only gallery distribution with direct sampling. Draw 1000 samples directly with `.sample(1000)` and 1000 samples with `MH`, and plot both over the density. Do the two sample sets look consistent?

 :::

In [ ]:
# your code here